# sgd-vanilla-from-scratch — worked example 2: SGD skips parameters with no gradient (frozen layers)

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `sgd-vanilla-from-scratch`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

In models with frozen layers or when some parameters did not participate in the forward pass, their `.grad` attribute remains None. The SGD step must skip these parameters rather than crash. This behavior mirrors PyTorch's optimizer — parameters with grad=None are silently left unchanged.

## Worked solution

**Step 1 — Create three parameters: two active, one frozen.** The frozen parameter has `requires_grad=False` (its grad will always be None).

**Step 2 — Set gradients only on the active parameters.** The frozen param stays at grad=None.

**Step 3 — Call sgd_step.** The loop uses `if p.grad is None: continue` to skip the frozen param.

**Step 4 — Check all three.** The active params updated, the frozen param is unchanged. We compare before/after values to confirm only the right ones moved.

In [ ]:
import torch as t

t.manual_seed(7)

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = array
        self.requires_grad = requires_grad
        self.grad = None
        self.recipe = None

def sgd_step(params, lr):
    for p in params:
        if p.grad is None:
            continue
        p.array -= lr * p.grad
        p.grad = None

lr = 0.05

p1 = MiniTensor(t.tensor([10.0]), requires_grad=True)
p2 = MiniTensor(t.tensor([5.0]),  requires_grad=True)
p3 = MiniTensor(t.tensor([2.0]),  requires_grad=False)  # frozen

p1.grad = t.tensor([4.0])
p2.grad = t.tensor([-2.0])
# p3.grad stays None

before = [p1.array.item(), p2.array.item(), p3.array.item()]
sgd_step([p1, p2, p3], lr)
after  = [p1.array.item(), p2.array.item(), p3.array.item()]

print('Before:', before)
print('After :', after)
print(f'p1: {before[0]} - {lr}*4 = {before[0]-lr*4:.3f} | got {after[0]:.3f}')
print(f'p2: {before[1]} - {lr}*(-2) = {before[1]-lr*(-2):.3f} | got {after[1]:.3f}')
print(f'p3 unchanged: {before[2]} == {after[2]} : {before[2] == after[2]}')